In [8]:
### Import necessary libs

from sympy import Function, dsolve, Eq, Derivative, symbols,cos
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from scipy.integrate import solve_ivp, odeint
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, Rectangle, Arrow
from matplotlib.collections import PatchCollection
from IPython.display import HTML
from scipy import signal

plt.style.use(['science',"grid"])
plt.rcParams['grid.color'] = 'grey'
plt.rcParams['grid.alpha'] = 0.6
plt.rcParams['grid.linestyle'] = '--'


# Define the independent variable
t = symbols('t')

# Define the dependent variable as a function of t
x = Function('x')

# Coefficients m,c,k and right-hand side F(t)
m, c, k, A, omega_F, x_0 = symbols('m c k A omega_F x_0')
F = symbols('F', cls=Function)

# The actual second-order ODE
ode = Eq( m*Derivative(x(t), t, t) + c*Derivative(x(t), t) + k*x(t), A*cos(omega_F*t))

# Solve ODE
solution = dsolve(ode, x(t),ics={x(0): x_0, Derivative(x(t), t).subs(t, 0): 0})
#sp.simplify(solution)

# Set specific values for parameter
argdict = {
    "m":5,
    "k":10,
    "c":0.5,
    "A" : 3,
    "omega_F":2
}
inidict = {
    "x_0":0.8,
    "xd_0":0
}


# define time-span and stepping
tsa = np.linspace(0,30,300)
# arguments and initial data
arg = [val for val in argdict.values()]
y0= np.array([val for val in inidict.values()])

def getAnalyticSolution(argdict,inidict,tsa):
        
    # transform symbolic function to callable method
    x_sol_a = sp.lambdify(t,solution.rhs.subs(argdict).subs(inidict))
    xa = x_sol_a(tsa)

    v_sol_a = sp.lambdify(t,sp.simplify(solution.rhs.subs(argdict).subs(inidict)).diff(t))
    va = v_sol_a(tsa)

    return xa,va,x_sol_a,v_sol_a

## External forcing $A cos(\omega t)$, set $A=0$ to make problem homogeneous
def forcing(t, A=0, omega=1):
    return A*np.cos(omega*t)

def forceNM(t):
    return np.array([forcing(t, argdict["A"], argdict["omega_F"])])

## Diffeq returns (x', x''), with algebraic expression for x'' solved above
def diffeq(t, u, m, k, b, A=0, omega=1):
    x = u[0]
    v = u[1]
    return np.array((v,-b/m*v-k/m*x+forcing(t, A, omega)/m))

## Jacobian der DGL
def jacdiffeq(t, u, m, k, b, A=0, omega=1):
    jac = np.zeros((np.shape(u)[0],np.shape(u)[0]))
    jac[0,0] = 0.0
    jac[0,1] = 1.0
    jac[1,0] = -k/m
    jac[1,1] = -b/m
    return jac

def energy(diffeq,t, u, m, k, b, A=0, omega=1):
    v,a = diffeq(t, u, m, k, b, A, omega)
    E_pot = 0.5*k*u[0]**2
    E_kin = 0.5*m*v**2
    E_tot = E_pot + E_kin
    return E_pot, E_kin, E_tot

def plotSolution(argdict,inidict,tsa,method,dt,ax,param=None):
    # arguments and initial data
    arg = [val for val in argdict.values()]
    y0= np.array([val for val in inidict.values()])
    
    xa,va,x_sol_a,v_sol_a = getAnalyticSolution(argdict,inidict,tsa)
    ax.clear()  # Clear the current axes
    t_end = tsa[-1]
    t_start = tsa[0]
    nsteps = int((t_end-t_start)/dt)
    ts = np.linspace(t_start,t_end,nsteps)
    
    y,E_pot,E_kin,E_tot= method(diffeq,y0,ts,arg)
    
    
    ax.plot(tsa,xa,label="analytic")
    ax.plot(ts,y,label=(method.__name__))
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)
    
    ax.set_xlabel("$t$")
    ax.set_ylabel("$x(t)$")
    
    ax.set_ylim([1.5*np.min(xa),1.5*np.max(xa)])

    
    plt.draw()  # Update the plot   

def plotSolution_impl(argdict,inidict,tsa,method,dt,ax,param=None):
    # arguments and initial data
    arg = [val for val in argdict.values()]
    y0= np.array([val for val in inidict.values()])
    
    xa,va,x_sol_a,v_sol_a = getAnalyticSolution(argdict,inidict,tsa)
    ax.clear()  # Clear the current axes
    t_end = tsa[-1]
    t_start = tsa[0]
    nsteps = int((t_end-t_start)/dt)
    ts = np.linspace(t_start,t_end,nsteps)
    
    y,E_pot,E_kin,E_tot= method(diffeq,jacdiffeq,y0,ts,arg)
    
    
    ax.plot(tsa,xa,label="analytic")
    ax.plot(ts,y,label=(method.__name__))
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)
    
    ax.set_xlabel("$t$")
    ax.set_ylabel("$x(t)$")
    
    ax.set_ylim([1.5*np.min(xa),1.5*np.max(xa)])

    
    plt.draw()  # Update the plot   
    
    
from ipywidgets import FloatSlider, interact,GridspecLayout


grid = GridspecLayout(3, 2)

# Widget to select dt
grid[0,0] = FloatSlider(value=0.1, min=0.001, max=1.0, step=0.001, description='Timestep (dt)')

grid[1,0] = FloatSlider(value=10, min=0.1, max=100.0, step=0.1, description='Stiffness k')

grid[2,0] = FloatSlider(value=5, min=0.1, max=100.0, step=0.1, description='Mass k')

grid[0,1] = FloatSlider(value=0.5, min=0.0, max=100.0, step=0.1, description='damping c')

grid[1,1] = FloatSlider(value=3, min=0.0, max=100.0, step=0.1, description='Force Amplitude A')

grid[2,1] = FloatSlider(value=2, min=0.0, max=100.0, step=0.1, description='Force frequency $\omega_F$')

# Function to update plot with new dt
def update_plot(dt,m,k,c,A,omega_F):
    fig, ax = plt.subplots(figsize=(9,6))
    argdict = {
    "m":m,
    "k":k,
    "c":c,
    "A" : A,
    "omega_F":omega_F
    }   
    plotSolution(argdict, inidict, tsa, integrator, dt, ax)

def update_plot_impl(dt,m,k,c,A,omega_F):

    argdict = {
    "m":m,
    "k":k,
    "c":c,
    "A" : A,
    "omega_F":omega_F
    }   
    plotSolution_impl(argdict, inidict, tsa, integrator, dt, ax)


def euler_vorwaerts(ode,y0,ts,arguments):
    """Euler Vorwärts Methode zur Lösung von System von DGL erster Ordnung
      ode: Funktion der DGL
      y0: Anfangswert
      ts: Zeitpunkte
      arguments : Zusätzliche Argumente für die DGL
    """
    y0t = np.array([y0]) if np.isscalar(y0) else y0
    y = np.zeros((len(ts),len(y0t)))
    E_pot = np.zeros((len(ts),1))
    E_kin = np.zeros((len(ts),1))
    E_tot = np.zeros((len(ts),1))
    E_pot[0],E_kin[0],E_tot[0] = energy(ode,ts[0], y0, *arguments)
    y[0,:] = y0t
    tn = ts[0]
    for i,t in enumerate(ts[1:]):
        dt = t-tn
        y[i+1,:] = y[i,:] + dt * ode(tn,y[i,:],*arguments) 
        E_pot[i+1],E_kin[i+1],E_tot[i+1] = energy(ode,t, y[i+1], *arguments)
        tn = t
    return y[:,0],E_pot,E_kin,E_tot


integrator = euler_vorwaerts
# Create interactive plot
interact(update_plot, dt=grid[0,0],m=grid[1,0],k=grid[2,0],c=grid[0,1],A=grid[1,1],omega_F=grid[2,1]); 

interactive(children=(FloatSlider(value=0.1, description='Timestep (dt)', layout=Layout(grid_area='widget001')…